# Multi-Query Demo：LangChain

Multi-Query Retriever 会让 LLM 从不同角度改写用户问题，再分别检索并合并结果，减少单个查询表达方式带来的召回遗漏。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader

load_dotenv()
knowledge_path = "../knowledge_db/prompt_engineering"
persist_path = "../vector_db/multi-query-langchain"
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)
embedding_model = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")

Multi-Query 检索器流程：

用户问题
→ LLM 改写成多个问题
→ 每个问题交给 base_retriever
→ 合并多个查询的结果
→ 去除重复文档
→ 返回最终文档列表

In [ ]:
documents = DirectoryLoader(
    knowledge_path,
    glob="**/*.md",
    loader_cls=UnstructuredMarkdownLoader,
).load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
).split_documents(documents)
vectorstore = Chroma.from_documents(chunks, embedding_model, persist_directory=persist_path)

# 基础检索器 负责一个问题 → 向量搜索 → 返回相似文档
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 创建的是一个新的 Multi-Query 检索器对象 “增强版检索器”
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
)

In [ ]:
question = "总结文本转换这篇文章的主要观点、方法和示例"
docs = retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    "只根据上下文回答问题，覆盖多个相关片段，不要只总结一个示例。"
    "\n上下文：{context}\n问题：{question}"
)
answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})
print(answer)
print("\n--- 检索来源 ---")
for doc in docs:
    print(doc.metadata.get("source"))

```python
[
    Document(page_content="相关内容1", metadata={...}),
    Document(page_content="相关内容2", metadata={...}),
]
```

```python
[
    Document(
        page_content="文本转换是对原始文本进行改写、翻译、格式转换或风格调整……",
        metadata={
            "source": "../knowledge_db/prompt_engineering/6. 文本转换 Transforming.md"
        }
    ),
    Document(
        page_content="常见的文本转换任务包括语言翻译、语气调整和格式转换……",
        metadata={
            "source": "../knowledge_db/prompt_engineering/6. 文本转换 Transforming.md"
        }
    ),
]
```

docs
└── list
    ├── Document
    │   ├── page_content: 文档文本
    │   └── metadata: 文件来源等信息
    ├── Document
    └── Document